# Physics-Guided Ignition Probability Modelling
## Using ERA5, MODIS, Sentinel, SRTM, and FIRMS

This notebook walks through the complete pipeline:
1. **Data Download** (synthetic for demo)
2. **Preprocessing & Gridding**
3. **Normalisation**
4. **Physics-Guided Feature Engineering**
5. **Model Training & Evaluation**
6. **Visualisation & Analysis**

---

## 0 · Setup & Imports

In [ ]:
import os, sys, yaml
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# Add project root to path
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)

# Load config
with open(os.path.join(ROOT, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

print('Study Region:', cfg['region']['name'])
print('Time Range:', cfg['time']['start'], '→', cfg['time']['end'])
print('Grid Resolution:', cfg['grid']['resolution'], '°')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

## 1 · Generate Synthetic Data

For demonstration, we generate synthetic datasets that mimic the structure of real ERA5, MODIS, SRTM, and FIRMS data. Replace this step with real data downloads when API keys are configured.

In [ ]:
from scripts.download_era5 import generate_synthetic_era5
from scripts.download_firms import generate_synthetic_firms
from scripts.download_modis import generate_synthetic_modis
from scripts.download_srtm import generate_synthetic_srtm

print('Generating synthetic ERA5 data...')
generate_synthetic_era5(cfg)

print('\nGenerating synthetic FIRMS data...')
generate_synthetic_firms(cfg)

print('\nGenerating synthetic MODIS data...')
generate_synthetic_modis(cfg)

print('\nGenerating synthetic SRTM slope data...')
generate_synthetic_srtm(cfg)

## 2 · Preprocessing & Gridding

Process raw data into daily gridded variables:
- **ERA5** → T_max, RH_min, U10_max, P_tot, SM_top
- **MODIS** → NDVI, NDWI (interpolated to daily)
- **SRTM** → slope (static)
- **FIRMS** → ignition labels y(t), FRP/count history features

In [ ]:
from src.preprocess import process_era5, process_modis, process_srtm, process_firms
from src.grid import grid_info

gi = grid_info(cfg)
raw_dir = os.path.join(ROOT, cfg['paths']['raw_data'])

print(f'Grid: {gi["n_lat"]} lat × {gi["n_lon"]} lon = {gi["n_lat"] * gi["n_lon"]} cells\n')

# Process each dataset
print('[ERA5]')
era5 = process_era5(raw_dir, cfg)
print(f'  → Shape: {dict(era5.dims)}')

print('\n[MODIS]')
modis = process_modis(raw_dir, cfg)
print(f'  → Shape: {dict(modis.dims)}')

print('\n[SRTM]')
slope = process_srtm(raw_dir, cfg)
print(f'  → Shape: {slope.shape}')

print('\n[FIRMS]')
firms = process_firms(raw_dir, cfg)
print(f'  → Shape: {dict(firms.dims)}')
print(f'  → Fire rate: {float(firms["ignition"].mean()):.4f}')

### 2.1 · Quick visual check of processed data

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ERA5 T_max – one sample day
era5['t_max'].isel(time=180).plot(ax=axes[0, 0], cmap='YlOrRd')
axes[0, 0].set_title('T_max (°C) – sample day')

# ERA5 RH_min
era5['rh_min'].isel(time=180).plot(ax=axes[0, 1], cmap='YlGnBu_r')
axes[0, 1].set_title('RH_min (%) – sample day')

# ERA5 Wind
era5['u10_max'].isel(time=180).plot(ax=axes[0, 2], cmap='Blues')
axes[0, 2].set_title('U10_max (m/s) – sample day')

# MODIS NDVI
modis_aligned = modis.reindex(time=firms.time.values, method='nearest')
modis_aligned['ndvi'].isel(time=180).plot(ax=axes[1, 0], cmap='Greens')
axes[1, 0].set_title('NDVI – sample day')

# SRTM Slope
slope.plot(ax=axes[1, 1], cmap='terrain')
axes[1, 1].set_title('Slope (°)')

# FIRMS ignition cumulative
firms['ignition'].sum(dim='time').plot(ax=axes[1, 2], cmap='hot_r')
axes[1, 2].set_title('Total fire days per cell')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'outputs', 'data_overview.png'), dpi=150)
plt.show()

## 3 · Normalisation

**Min-max normalisation** to [0, 1] for each continuous driver, using climatological min/max over the full multi-year period:

$$\tilde{X} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

In [ ]:
from src.normalize import compute_norm_params, normalize, save_params

# Align time axes
common_times = firms.time.values
era5_aligned = era5.reindex(time=common_times, method='nearest')
modis_aligned = modis.reindex(time=common_times, method='nearest')

# Collect raw drivers
raw_drivers = {
    't_max':      era5_aligned['t_max'],
    'rh_min':     era5_aligned['rh_min'],
    'u10_max':    era5_aligned['u10_max'],
    'sm_top':     era5_aligned['sm_top'],
    'ndvi':       modis_aligned['ndvi'],
    'ndwi':       modis_aligned['ndwi'],
    'slope':      slope,
    'frp_hist':   firms['frp_hist'],
    'count_hist': firms['count_hist'],
}

variables = list(raw_drivers.keys())

# Compute normalisation parameters
print('Computing normalisation parameters:\n')
norm_params = compute_norm_params(raw_drivers, variables)

# Save for later inference
processed_dir = os.path.join(ROOT, cfg['paths']['processed_data'])
os.makedirs(processed_dir, exist_ok=True)
save_params(norm_params, os.path.join(processed_dir, 'norm_params.json'))

# Normalise
normed = {}
print('\nNormalised ranges:')
for var in variables:
    normed[var] = normalize(raw_drivers[var], norm_params[var])
    print(f'  {var:>12s} → [{float(np.nanmin(normed[var])):.3f}, {float(np.nanmax(normed[var])):.3f}]')

## 4 · Physics-Guided Feature Engineering

### Composite Factors

| Factor | Formula | Physical Meaning |
|--------|---------|------------------|
| $F_{\text{avail}}$ | $\tilde{NDVI}(t)$ | Live fuel biomass available |
| $F_{\text{dry}}$ | $\alpha_1(1-\tilde{NDWI}) + \alpha_2(1-\tilde{RH}) + \alpha_3(1-\tilde{SM})$ | Fuel dryness (LFMC proxy) |
| $G_{\text{spread}}$ | $1 + \beta_1 \tilde{U} + \beta_2 \tilde{\theta}$ | Wind–slope spread potential |
| $H_{\text{history}}$ | $\gamma_1 \tilde{FRP}_{\text{hist}} + \gamma_2 \tilde{Count}_{\text{hist}}$ | Recent fire activity |

### Composite Risk Index

$$R_{\text{phys}}(t) = \tilde{T}(t) \cdot F_{\text{avail}}(t) \cdot F_{\text{dry}}(t) \cdot G_{\text{spread}}(t) + H_{\text{history}}(t)$$

In [ ]:
from src.features import (
    compute_F_avail, compute_F_dry, compute_G_spread,
    compute_H_history, compute_R_phys, build_all_features
)

# Print physics weights
print('Physics weights from config:')
print(f'  α: {cfg["alpha"]}')
print(f'  β: {cfg["beta"]}')
print(f'  γ: {cfg["gamma"]}\n')

# Compute all features
features = build_all_features(normed, cfg)

print('Feature ranges:')
for name, arr in features.items():
    arr_np = np.asarray(arr)
    print(f'  {name:>12s}  range=[{float(np.nanmin(arr_np)):.4f}, {float(np.nanmax(arr_np)):.4f}]')

In [ ]:
# Visualise R_phys for a sample day (mid-summer)
if hasattr(features['R_phys'], 'dims') and 'time' in features['R_phys'].dims:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    features['R_phys'].isel(time=180).plot(
        ax=axes[0], cmap='YlOrRd', vmin=0
    )
    axes[0].set_title('R_phys – Sample Summer Day')
    
    # Mean R_phys over time
    features['R_phys'].mean(dim=['latitude', 'longitude']).plot(ax=axes[1])
    axes[1].set_title('R_phys – Spatial Mean Over Time')
    axes[1].set_ylabel('Mean R_phys')
    
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'outputs', 'R_phys_overview.png'), dpi=150)
    plt.show()
else:
    print('R_phys does not have time dimension – check feature computation')

## 5 · Dataset Assembly & Train/Test Split

**Temporal split** (no leakage):
- **Train**: 2021
- **Validation**: 2022
- **Test**: 2023

In [ ]:
# Build combined feature dataset
ds = xr.Dataset(
    coords={'time': common_times, 'latitude': gi['lats'], 'longitude': gi['lons']}
)

# Add normalised raw drivers
for var in variables:
    if hasattr(normed[var], 'dims') and 'time' in normed[var].dims:
        ds[var] = normed[var]
    else:
        ds[var] = (['latitude', 'longitude'], np.asarray(normed[var]))

# Add physics features
for name, arr in features.items():
    if hasattr(arr, 'dims') and 'time' in arr.dims:
        ds[name] = arr
    else:
        ds[name] = (['latitude', 'longitude'], np.asarray(arr))

# Add ignition label
ds['ignition'] = firms['ignition']

# Save features
features_dir = os.path.join(ROOT, cfg['paths']['features'])
os.makedirs(features_dir, exist_ok=True)
ds.to_netcdf(os.path.join(features_dir, 'features.nc'))
print(f'Features saved: {dict(ds.dims)}')
print(f'Variables: {list(ds.data_vars)}')

In [ ]:
from src.dataset import build_dataset, split_by_year, get_Xy

# Flatten to DataFrame
df = build_dataset(features_dir)
print(f'\nTotal samples: {len(df):,}')
print(f'Fire rate: {df["ignition"].mean()*100:.2f}%')
print(f'\nColumns: {list(df.columns)}')

# Split
train_df, val_df, test_df = split_by_year(
    df, cfg['time']['train_years'], cfg['time']['val_years'], cfg['time']['test_years']
)

print(f'\nTrain: {len(train_df):,} samples ({cfg["time"]["train_years"]})')
print(f'Val:   {len(val_df):,} samples ({cfg["time"]["val_years"]})')
print(f'Test:  {len(test_df):,} samples ({cfg["time"]["test_years"]})')

## 6 · Model Training

### 6.1 Physics-Guided Logistic Regression

$$p_{\text{ign}}(t) = \sigma(a \cdot R_{\text{phys}}(t) + b) = \frac{1}{1 + \exp(-(a \cdot R_{\text{phys}}(t) + b))}$$

In [ ]:
from src.models import (
    train_physics_logistic, train_attribute_logistic,
    train_random_forest, predict_proba, save_model
)

# Feature sets
raw_feature_names = ['t_max', 'rh_min', 'u10_max', 'sm_top',
                     'ndvi', 'ndwi', 'slope', 'frp_hist', 'count_hist']
available_raw = [f for f in raw_feature_names if f in df.columns]

# ── Model 1: Physics-Guided Logistic ──
print('═' * 50)
print('Model 1: Physics-Guided Logistic Regression')
print('═' * 50)

X_train, y_train = get_Xy(train_df, 'R_phys')
X_val, y_val     = get_Xy(val_df, 'R_phys')
X_test, y_test   = get_Xy(test_df, 'R_phys')

model_phys = train_physics_logistic(X_train, y_train)

# Learned parameters
a = model_phys.coef_[0, 0]
b = model_phys.intercept_[0]
print(f'  Learned: a = {a:.4f}, b = {b:.4f}')
print(f'  p_ign(R=0) = {1/(1+np.exp(-b)):.4f}')
print(f'  p_ign(R=1) = {1/(1+np.exp(-(a+b))):.4f}')

In [ ]:
# ── Model 2: Attribute-Only Logistic ──
print('═' * 50)
print('Model 2: Attribute-Only Logistic Regression')
print('═' * 50)

X_train_a, y_train_a = get_Xy(train_df, available_raw)
X_val_a, y_val_a     = get_Xy(val_df, available_raw)
X_test_a, y_test_a   = get_Xy(test_df, available_raw)

X_train_a = np.nan_to_num(X_train_a, nan=0.0)
X_val_a   = np.nan_to_num(X_val_a, nan=0.0)
X_test_a  = np.nan_to_num(X_test_a, nan=0.0)

model_attr = train_attribute_logistic(X_train_a, y_train_a)

# Feature importance
print('\nFeature coefficients:')
for name, coef in zip(available_raw, model_attr.coef_[0]):
    print(f'  {name:>12s}: {coef:+.4f}')

In [ ]:
# ── Model 3: Random Forest ──
print('═' * 50)
print('Model 3: Random Forest Baseline')
print('═' * 50)

model_rf = train_random_forest(X_train_a, y_train_a, n_estimators=200, max_depth=10)

# Feature importance
print('\nFeature importances:')
for name, imp in sorted(zip(available_raw, model_rf.feature_importances_),
                         key=lambda x: -x[1]):
    print(f'  {name:>12s}: {imp:.4f}')

## 7 · Evaluation

### 7.1 Core Metrics

In [ ]:
from src.evaluate import compute_metrics, print_metrics, compare_models

# Predictions on test set
y_prob_phys = predict_proba(model_phys, X_test)
y_prob_attr = predict_proba(model_attr, X_test_a)
y_prob_rf   = predict_proba(model_rf, X_test_a)

# Compute metrics
results = {
    'Physics Logistic':   compute_metrics(y_test, y_prob_phys),
    'Attribute Logistic': compute_metrics(y_test_a, y_prob_attr),
    'Random Forest':      compute_metrics(y_test_a, y_prob_rf),
}

# Print all
for name, m in results.items():
    print_metrics(name, m)

# Comparison table
comp_df = compare_models(results)
comp_df

### 7.2 ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ROC
for name, y_p, y_t in [
    ('Physics Logistic', y_prob_phys, y_test),
    ('Attribute Logistic', y_prob_attr, y_test_a),
    ('Random Forest', y_prob_rf, y_test_a),
]:
    fpr, tpr, _ = roc_curve(y_t, y_p)
    auc = roc_auc_score(y_t, y_p)
    ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

# PR
for name, y_p, y_t in [
    ('Physics Logistic', y_prob_phys, y_test),
    ('Attribute Logistic', y_prob_attr, y_test_a),
    ('Random Forest', y_prob_rf, y_test_a),
]:
    prec, rec, _ = precision_recall_curve(y_t, y_p)
    ap = average_precision_score(y_t, y_p)
    ax2.plot(rec, prec, label=f'{name} (AP={ap:.3f})', linewidth=2)

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'outputs', 'roc_pr_curves.png'), dpi=150)
plt.show()

### 7.3 Reliability Diagram

In [ ]:
from src.evaluate import reliability_diagram

reliability_diagram(
    y_test, y_prob_phys,
    save_path=os.path.join(ROOT, 'outputs', 'reliability_diagram.png')
)

# Also show inline
from IPython.display import Image
Image(os.path.join(ROOT, 'outputs', 'reliability_diagram.png'))

### 7.4 Model Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_list = ['auc_roc', 'auc_pr', 'brier_score']
titles = ['AUC-ROC ↑', 'AUC-PR ↑', 'Brier Score ↓']
colors = ['#2196F3', '#4CAF50', '#FF9800']

for ax, metric, title, color in zip(axes, metrics_list, titles, colors):
    vals = [results[m][metric] for m in results]
    bars = ax.bar(results.keys(), vals, color=color, alpha=0.7, edgecolor='black')
    ax.set_title(title, fontsize=13)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.2, axis='y')
    # Add value labels
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10)

plt.suptitle('Model Comparison on Test Set (2023)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'outputs', 'model_comparison.png'), dpi=150)
plt.show()

## 8 · Ignition Probability Map

Visualise the model output for a specific date.

In [ ]:
import matplotlib.colors as mcolors

# Choose a summer day from the test set
target_date = '2023-07-15'

# Get R_phys for that day
day_mask = test_df['time'] == pd.Timestamp(target_date)
if day_mask.sum() > 0:
    day_data = test_df[day_mask]
    X_day = day_data[['R_phys']].values
    probs = predict_proba(model_phys, X_day)
    
    # Reshape to grid
    prob_grid = np.full((gi['n_lat'], gi['n_lon']), np.nan)
    for _, row in day_data.iterrows():
        li = np.argmin(np.abs(gi['lats'] - row['latitude']))
        lo = np.argmin(np.abs(gi['lons'] - row['longitude']))
        idx = day_data.index.get_loc(row.name) if hasattr(day_data.index, 'get_loc') else 0
    
    # Alternative: rebuild from full feature grid
    features_ds = xr.open_dataset(os.path.join(features_dir, 'features.nc'))
    t_idx = np.argmin(np.abs(features_ds.time.values - np.datetime64(target_date)))
    R_day = features_ds['R_phys'].isel(time=t_idx).values
    
    prob_grid = 1 / (1 + np.exp(-(a * R_day + b)))
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    colors_cmap = ['#2d6a4f', '#95d5b2', '#ffff3f', '#ff9f1c', '#e63946']
    cmap = mcolors.LinearSegmentedColormap.from_list('fire_risk', colors_cmap)
    
    im = ax.pcolormesh(gi['lons'], gi['lats'], prob_grid,
                       cmap=cmap, vmin=0, vmax=1, shading='auto')
    fig.colorbar(im, ax=ax, shrink=0.7, label='p_ign')
    ax.set_title(f'Ignition Probability – {target_date}', fontsize=14)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')
    ax.text(-119, 37, 'California', fontsize=12, ha='center', alpha=0.5, style='italic')
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'outputs', f'prob_map_{target_date}.png'), dpi=150)
    plt.show()
else:
    print(f'No data for {target_date} in test set')

## 9 · Save Models

In [ ]:
models_dir = os.path.join(ROOT, cfg['paths']['models'])
save_model(model_phys, models_dir, 'physics_logistic')
save_model(model_attr, models_dir, 'attribute_logistic')
save_model(model_rf, models_dir, 'random_forest')

# Save comparison table
comp_df.to_csv(os.path.join(ROOT, 'outputs', 'model_comparison.csv'))
print('\nAll models and results saved!')

## 10 · Summary

### Key Findings
- The physics-guided logistic regression uses a single composite index $R_{\text{phys}}$ to predict ignition probability
- The multiplicative structure mirrors established fire weather indices (FWI)
- Comparison with attribute-only logistic and Random Forest baselines shows the value of physics-guided feature engineering

### Model Equation
$$p_{\text{ign}}(t) = \sigma(a \cdot R_{\text{phys}}(t) + b)$$

### Next Steps
- Download and process real ERA5/MODIS/FIRMS data
- Tune physics weights (α, β, γ) via cross-validation
- Add spatial diagnostics by land cover / slope class
- Consider GAM or neural ODE extensions for future work